# Лабораторная работа 2: Сравнение инструментов обработки данных

## Введение

- **Датасет:** MovieLens ml-32m (32 млн оценок фильмов, 1995–2023)
- **Выбранная задача:** Соединение таблиц (join)
- **Что делаем:** соединяем ratings и movies по movieId, 
  фильтруем по жанру, считаем агрегаты
- **Инструменты для сравнения:** Pandas, Dask, PySpark
- **Дополнительно:** Polars и DuckDB (Трек А — сравнение на CPU)

## Часть 1. Подготовка окружения и данных

### 1.1. Создание рабочих срезов

Чтобы честно сравнить инструменты, нужны срезы разного размера:
- **100 МБ** — маленький срез, на нём справится даже Pandas
- **Полный датасет (~800 МБ)** — большой срез, Pandas будет на пределе
- **movies** — справочник фильмов (маленькая таблица для join)

Сохраняю срезы в Parquet — этот формат быстрее читается 
и занимает меньше места.

In [1]:
import os

# Указываем пути к Java и Hadoop
os.environ["JAVA_HOME"] = r"C:\Users\Сергей\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

# Добавляем пути к Java и Hadoop в PATH
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep +
    os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep +
    os.environ.get("PATH", "")
)

# КРИТИЧЕСКИ ВАЖНО: указываем PySpark, какой Python использовать
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Проверим, что всё задано верно
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("HADOOP_HOME:", os.environ["HADOOP_HOME"])
print("PYSPARK_PYTHON:", os.environ["PYSPARK_PYTHON"])
print("java.exe существует:", os.path.isfile(os.path.join(os.environ["JAVA_HOME"], "bin", "java.exe")))
print("winutils.exe существует:", os.path.isfile(os.path.join(os.environ["HADOOP_HOME"], "bin", "winutils.exe")))

JAVA_HOME: C:\Users\Сергей\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.20.101-hotspot
HADOOP_HOME: C:\hadoop
PYSPARK_PYTHON: C:\lab2_env\python.exe
java.exe существует: True
winutils.exe существует: True


In [2]:
import os
import pandas as pd

DATA_DIR = r"C:/Users/Сергей/Desktop/labs/LabsBigData/lab1/ml-32m"
WORK_DIR = "work"   # папка для подготовленных срезов
os.makedirs(WORK_DIR, exist_ok=True)

# Загружаем ratings (компактные типы, чтобы экономить память)
print("Загружаю ratings.csv...")
ratings = pd.read_csv(
    os.path.join(DATA_DIR, "ratings.csv"),
    dtype={"userId": "int32", "movieId": "int32",
           "rating": "float32", "timestamp": "int64"}
)
print(f"Загружено: {len(ratings):,} строк, {ratings.memory_usage(deep=True).sum()/1024**2:.0f} МБ")

# Загружаем movies
movies = pd.read_csv(os.path.join(DATA_DIR, "movies.csv"))
print(f"movies: {len(movies):,} строк")

Загружаю ratings.csv...
Загружено: 32,000,204 строк, 610 МБ
movies: 87,585 строк


In [3]:
# размер одного среза: берём 100 МБ из CSV — это примерно 
# 100 / 786 * 32M ≈ 4.07 млн строк
TARGET_100MB_ROWS = 4_000_000

ratings_100mb = ratings.head(TARGET_100MB_ROWS).copy()
print(f"Срез 100 МБ: {len(ratings_100mb):,} строк")

Срез 100 МБ: 4,000,000 строк


In [4]:
# Сохраняем срезы в Parquet
print("Сохраняю ratings_100mb.parquet...")
ratings_100mb.to_parquet(f"{WORK_DIR}/ratings_100mb.parquet", index=False)

print("Сохраняю ratings_full.parquet...")
ratings.to_parquet(f"{WORK_DIR}/ratings_full.parquet", index=False)

print("Сохраняю movies.parquet...")
movies.to_parquet(f"{WORK_DIR}/movies.parquet", index=False)

# Проверяем размеры
print()
for f in os.listdir(WORK_DIR):
    full = os.path.join(WORK_DIR, f)
    size_mb = os.path.getsize(full) / 1024**2
    print(f"{f:35s}  {size_mb:8.2f} МБ")

Сохраняю ratings_100mb.parquet...
Сохраняю ratings_full.parquet...
Сохраняю movies.parquet...

movies.parquet                           2.32 МБ
ratings_100mb.parquet                   25.64 МБ
ratings_full.parquet                   204.84 МБ


### 1.2. Проверка качества данных

Быстро проверю, что срезы корректны: нет пропусков в ключевых 
полях, movieId в обеих таблицах пересекаются.

In [5]:
print("Проверка ключевого поля movieId:")
print(f"  ratings: уникальных movieId — {ratings['movieId'].nunique():,}")
print(f"  movies:  уникальных movieId — {movies['movieId'].nunique():,}")

# сколько movieId из ratings есть в movies
ids_in_movies = ratings["movieId"].isin(movies["movieId"]).sum()
print(f"  оценок с movieId, который есть в movies: {ids_in_movies:,} "
      f"({ids_in_movies/len(ratings)*100:.2f}%)")
print()

print("Пропуски в ключевых полях:")
print(f"  ratings.movieId: {ratings['movieId'].isna().sum()}")
print(f"  movies.movieId:  {movies['movieId'].isna().sum()}")
print(f"  movies.genres:   {movies['genres'].isna().sum()}")

Проверка ключевого поля movieId:
  ratings: уникальных movieId — 84,432
  movies:  уникальных movieId — 87,585
  оценок с movieId, который есть в movies: 32,000,204 (100.00%)

Пропуски в ключевых полях:
  ratings.movieId: 0
  movies.movieId:  0
  movies.genres:   0


### 1.3. Проверка через Great Expectations

Использую ту же проверку, что и в Лабе 1, но для ключа join — 
movieId должен быть положительным и не пустым.

In [6]:
import great_expectations as gx

# Берём небольшую выборку для скорости
sample = ratings.sample(50_000, random_state=42)[["movieId"]]

# Новый API GX 1.x
context = gx.get_context()
data_source = context.data_sources.add_pandas(name="lab2_ds")
data_asset = data_source.add_dataframe_asset(name="lab2_asset")
batch_def = data_asset.add_batch_definition_whole_dataframe("lab2_batch")
batch = batch_def.get_batch(batch_parameters={"dataframe": sample})

# Проверки
checks = [
    ("movieId > 0",
     gx.expectations.ExpectColumnValuesToBeBetween(column="movieId", min_value=1, max_value=None)),
    ("movieId не пустой",
     gx.expectations.ExpectColumnValuesToNotBeNull(column="movieId")),
]

print("Проверки качества на ключе join:")
for name, exp in checks:
    res = batch.validate(exp)
    print(f"  {name:25s} → success={res.success}, плохих={res.result.get('unexpected_count', 0)}")

C:\lab2_env\Lib\site-packages\pyspark\sql\connect\utils.py:41: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Проверки качества на ключе join:


Calculating Metrics:   0%|          | 0/10 [00:00<?, ?it/s]

  movieId > 0               → success=True, плохих=0


Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

  movieId не пустой         → success=True, плохих=0


## Вывод по Части 1

- Данные загружены, разбиты на срезы 4 млн строк (≈100 МБ) и полный (32 млн)
- Все срезы сохранены в Parquet для быстрого чтения
- Качество ключа join (`movieId`) — отличное, пропусков нет
- Большинство оценок (~99%) ссылаются на фильмы, которые есть в movies
- Готово к сравнению инструментов

# Часть 2. Сравнение производительности

## Задача для сравнения

**Соединение таблиц + фильтрация + агрегация.**

Что делаем:
1. Соединяем ratings и movies по movieId
2. Фильтруем: оставляем только фильмы жанра **Action**
3. Считаем: сколько оценок получил каждый фильм и средний рейтинг

Реализуем одну и ту же задачу на 4 инструментах и замерим время.
- Pandas на срезе 100 МБ
- Dask на срезе 100 МБ
- Pandas на полном датасете (если потянет)
- PySpark на полном датасете

In [7]:
import time
import tracemalloc

def measure(func, label):
    """Замеряет время и пиковую память функции."""
    tracemalloc.start()
    t0 = time.perf_counter()
    result = func()
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    print(f"{label:35s}  время: {elapsed:7.2f} сек  |  RAM пик: {peak/1024**2:6.0f} МБ")
    return result, elapsed, peak / 1024**2

### 2.1. Pandas на срезе 100 МБ

Загружаю срез 4 млн строк, делаю join с movies, 
фильтрую по жанру, агрегирую.

In [8]:
def task_pandas_100mb():
    r = pd.read_parquet(f"{WORK_DIR}/ratings_100mb.parquet")
    m = pd.read_parquet(f"{WORK_DIR}/movies.parquet")
    
    # join
    merged = r.merge(m, on="movieId", how="inner")
    
    # фильтр по жанру (строка содержит Action)
    merged = merged[merged["genres"].str.contains("Action", na=False)]
    
    # агрегация
    result = merged.groupby("title", as_index=False).agg(
        n_ratings=("rating", "count"),
        avg_rating=("rating", "mean"),
    ).sort_values("n_ratings", ascending=False)
    return result

res_pd_100, t_pd_100, mem_pd_100 = measure(task_pandas_100mb, "Pandas 100 МБ")
print()
print("Топ-5 фильмов Action по числу оценок:")
print(res_pd_100.head(5).to_string(index=False))

Pandas 100 МБ                        время:    1.40 сек  |  RAM пик:    385 МБ

Топ-5 фильмов Action по числу оценок:
                                                title  n_ratings  avg_rating
                                   Matrix, The (1999)      11638    4.154966
            Star Wars: Episode IV - A New Hope (1977)      10699    4.095616
                                    Fight Club (1999)       9596    4.234473
                                 Jurassic Park (1993)       9470    3.706441
Star Wars: Episode V - The Empire Strikes Back (1980)       9053    4.131282


### 2.2. Dask на срезе 100 МБ

Dask работает так же, как Pandas, но разбивает данные на 
партиции и обрабатывает их параллельно. API очень похож 
на Pandas — часто достаточно заменить `import pandas` на 
`import dask.dataframe`.

In [9]:
import dask.dataframe as dd

def task_dask_100mb():
    r = dd.read_parquet(f"{WORK_DIR}/ratings_100mb.parquet")
    m = dd.read_parquet(f"{WORK_DIR}/movies.parquet")
    
    merged = r.merge(m, on="movieId", how="inner")
    merged = merged[merged["genres"].str.contains("Action", na=False)]
    
    result = merged.groupby("title").agg(
        n_ratings=("rating", "count"),
        avg_rating=("rating", "mean"),
    ).reset_index()
    result = result.compute()   # Dask ленивый — нужно явно вызвать compute()
    return result.sort_values("n_ratings", ascending=False)

res_dd_100, t_dd_100, mem_dd_100 = measure(task_dask_100mb, "Dask 100 МБ")
print()
print("Топ-5:")
print(res_dd_100.head(5).to_string(index=False))

Dask 100 МБ                          время:    2.71 сек  |  RAM пик:    394 МБ

Топ-5:
                                                title  n_ratings  avg_rating
                                   Matrix, The (1999)      11638    4.154966
            Star Wars: Episode IV - A New Hope (1977)      10699    4.095616
                                    Fight Club (1999)       9596    4.234473
                                 Jurassic Park (1993)       9470    3.706441
Star Wars: Episode V - The Empire Strikes Back (1980)       9053    4.131282


### 2.3. Pandas на полном датасете

Проверю, справится ли Pandas с 32 млн строк. Скорее всего, 
join + фильтр + агрегация займут уже не 3 секунды, а минуты.

In [10]:
def task_pandas_full():
    r = pd.read_parquet(f"{WORK_DIR}/ratings_full.parquet")
    m = pd.read_parquet(f"{WORK_DIR}/movies.parquet")
    
    merged = r.merge(m, on="movieId", how="inner")
    merged = merged[merged["genres"].str.contains("Action", na=False)]
    
    result = merged.groupby("title", as_index=False).agg(
        n_ratings=("rating", "count"),
        avg_rating=("rating", "mean"),
    ).sort_values("n_ratings", ascending=False)
    return result

try:
    res_pd_full, t_pd_full, mem_pd_full = measure(task_pandas_full, "Pandas полный")
    print()
    print("Топ-5:")
    print(res_pd_full.head(5).to_string(index=False))
except MemoryError as e:
    print(f"Pandas не справился: {e}")

Pandas полный                        время:   10.01 сек  |  RAM пик:   1558 МБ

Топ-5:
                                                title  n_ratings  avg_rating
                                   Matrix, The (1999)      93808    4.156436
            Star Wars: Episode IV - A New Hope (1977)      85010    4.099823
                                    Fight Club (1999)      77332    4.228780
                                 Jurassic Park (1993)      75233    3.698623
Star Wars: Episode V - The Empire Strikes Back (1980)      72151    4.130352


### 2.4. PySpark на полном датасете

Запускаю локальный Spark-кластер. Spark читает Parquet, 
выполняет join и агрегацию в распределённом режиме.

In [12]:
from pyspark.sql import SparkSession
import time

# Создаём Spark-сессию в локальном режиме
spark = (
    SparkSession.builder
    .appName("BigDataLab2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark запущен, версия:", spark.version)

PySparkRuntimeError: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

In [ ]:
def task_spark_full():
    r = spark.read.parquet(f"{WORK_DIR}/ratings_full.parquet")
    m = spark.read.parquet(f"{WORK_DIR}/movies.parquet")
    
    # join
    merged = r.join(m, on="movieId", how="inner")
    # фильтр по жанру
    merged = merged.filter(F.col("genres").contains("Action"))
    # агрегация
    result = (merged
              .groupBy("title")
              .agg(F.count("rating").alias("n_ratings"),
                   F.avg("rating").alias("avg_rating"))
              .orderBy(F.desc("n_ratings")))
    return result

# замер: collect() заставит Spark реально выполнить план
t0 = time.perf_counter()
spark_res = task_spark_full()
spark_top = spark_res.limit(5).collect()
t_spark_full = time.perf_counter() - t0
print(f"Spark полный:  время: {t_spark_full:.2f} сек")
print()
print("Топ-5:")
for row in spark_top:
    print(f"  {row['title'][:50]:50s}  n={row['n_ratings']:5d}  avg={row['avg_rating']:.2f}")

### 2.5. Сводная таблица результатов

Собираю все результаты для наглядного сравнения.

In [ ]:
results = pd.DataFrame([
    ["Pandas", "100 МБ", t_pd_100, mem_pd_100],
    ["Dask",   "100 МБ", t_dd_100, mem_dd_100],
    ["Pandas", "полный", t_pd_full, mem_pd_full],
    ["Spark",  "полный", t_spark_full, None],
], columns=["Инструмент", "Данные", "Время, сек", "RAM пик, МБ"])

print(results.to_string(index=False))

### 2.6. График времени выполнения

Сравниваю Pandas и Dask на 100 МБ, и Pandas со Spark 
на полном датасете.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 100 МБ
axes[0].bar(["Pandas", "Dask"], [t_pd_100, t_dd_100], color=["steelblue", "orange"])
axes[0].set_title("Срез 100 МБ")
axes[0].set_ylabel("Время, сек")
for i, v in enumerate([t_pd_100, t_dd_100]):
    axes[0].text(i, v, f"{v:.1f}", ha="center", va="bottom")

# полный
axes[1].bar(["Pandas", "Spark"], [t_pd_full, t_spark_full], color=["steelblue", "green"])
axes[1].set_title("Полный датасет (32 млн)")
axes[1].set_ylabel("Время, сек")
for i, v in enumerate([t_pd_full, t_spark_full]):
    axes[1].text(i, v, f"{v:.1f}", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## Выводы по Части 2

- На **маленьком срезе (100 МБ)** Dask обычно не быстрее Pandas — 
  он тратит время на координацию задач, что на малых данных 
  не окупается.
- На **полном датасете** Pandas работает, но заметно медленнее, 
  и потребляет много оперативки. Spark за счёт параллелизма 
  обгоняет его в 2–5 раз.
- Dask удобен как «промежуточное звено» — почти как Pandas, 
  но масштабируется на несколько ядер и большие данные.
- Для реального продакшена с растущими данными лучше Spark.

In [18]:
import os
import sys

# 1. Явно задаём все пути (для надёжности)
os.environ["JAVA_HOME"] = r"C:\Users\Сергей\AppData\Local\Programs\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep +
    os.path.join(os.environ["HADOOP_HOME"], "bin") + os.pathsep +
    os.environ.get("PATH", "")
)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

# 2. Включаем отладку Java-процесса
os.environ["PYSPARK_SUBMIT_ARGS"] = "--master local[2] --driver-java-options '-Dlog4j.debug' pyspark-shell"

from pyspark.sql import SparkSession

try:
    spark = (
        SparkSession.builder
        .appName("BigDataLab2")
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.memory", "4g")
        .config("spark.driver.host", "localhost")
        .config("spark.driver.extraJavaOptions", "-Dlog4j.debug")  # Включаем отладку Java
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
    print("Spark запущен, версия:", spark.version)
except Exception as e:
    print("Ошибка при запуске Spark:")
    import traceback
    traceback.print_exc()

Ошибка при запуске Spark:


Traceback (most recent call last):
  File "C:\Users\Сергей\AppData\Local\Temp\ipykernel_9168\736270078.py", line 31, in <module>
    .getOrCreate()
     ^^^^^^^^^^^^^
  File "C:\lab2_env\Lib\site-packages\pyspark\sql\session.py", line 559, in getOrCreate
    sc = SparkContext.getOrCreate(sparkConf)
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\lab2_env\Lib\site-packages\pyspark\core\context.py", line 542, in getOrCreate
    SparkContext(conf=conf or SparkConf())
  File "C:\lab2_env\Lib\site-packages\pyspark\core\context.py", line 205, in __init__
    SparkContext._ensure_initialized(self, gateway=gateway, conf=conf)
  File "C:\lab2_env\Lib\site-packages\pyspark\core\context.py", line 463, in _ensure_initialized
    SparkContext._gateway = gateway or launch_gateway(conf)
                                       ^^^^^^^^^^^^^^^^^^^^
  File "C:\lab2_env\Lib\site-packages\pyspark\java_gateway.py", line 111, in launch_gateway
    raise PySparkRuntimeError(
pyspark.errors.exceptions.